In [ ]:
# ==============================================================================# 🚀 DO NOT MODIFY: Standardized Notebook Setup# ==============================================================================# This cell is designed to work in both Google Colab and local environments.# It ensures that the environment is correctly configured by cloning (or# locating) the project repository and installing the necessary dependencies.## ------------------------------------------------------------------------------##  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):##  This cell will automatically find the repository root and configure your#  environment. Just make sure you have run: pip install -e .[dev]## ------------------------------------------------------------------------------import importlib.utilimport osimport subprocessimport sysfrom pathlib import Path# --- Configuration ---REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned# --- End of Configuration ---def find_repo_root(start_path: Path) -> Path | None:    """    Find the repository root by looking for pyproject.toml.    Searches upward from start_path until it finds pyproject.toml or hits root.    Args:        start_path: Directory to start searching from.    Returns:        Path to repository root, or None if not found.    """    current = start_path.resolve()    while current != current.parent:  # Stop at filesystem root        if (current / "pyproject.toml").exists():            return current        current = current.parent    return Nonedef detect_active_branch(repo_dir: Path) -> str:    """    Determine the active git branch for pulling updates.    Tries multiple methods to detect the current branch name.    Args:        repo_dir: Path to the git repository.    Returns:        Branch name (defaults to 'master' if detection fails).    """    commands = [        "git symbolic-ref --short HEAD",        "git rev-parse --abbrev-ref HEAD",    ]    for cmd in commands:        result = subprocess.run(            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True        )        if result.returncode == 0:            branch = result.stdout.strip()            if branch and not branch.startswith("origin/"):                return branch    return "master"def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:    """    Run a shell command and raise an error if it fails.    Args:        cmd: The command to run.        cwd: Optional working directory for the command.    Raises:        RuntimeError: If the command returns a non-zero exit code.    """    result = subprocess.run(cmd, shell=True, cwd=cwd)    if result.returncode != 0:        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")def load_setup_module(repo_path: Path):    """Load the setup module directly without triggering package imports."""    setup_path = repo_path / "core" / "notebook" / "setup.py"    spec = importlib.util.spec_from_file_location("_setup_module", setup_path)    setup_module = importlib.util.module_from_spec(spec)    spec.loader.exec_module(setup_module)    return setup_module# --- Detect environment ---try:    import google.colab  # noqa: F401    IN_COLAB = Trueexcept ImportError:    IN_COLAB = False# --- Main setup logic ---if IN_COLAB:    print("☁️  Running in Google Colab. Setting up the environment...\n")    # Determine repository path    start_dir = Path.cwd()    if start_dir.name == REPO_DIR.name:        repo_path = start_dir    else:        repo_path = start_dir / REPO_DIR    # Clone or update repository    if not repo_path.exists():        print(f"📥 Cloning repository from {REPO_URL}...")        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")        print(f"✅ Repository cloned to {repo_path}\n")    else:        print(f"📂 Repository already exists at {repo_path}")        active_branch = detect_active_branch(repo_path)        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)        print(f"✅ Repository updated\n")    # Verify repository structure    if not (repo_path / "pyproject.toml").exists():        raise FileNotFoundError(            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "            "The repository may be corrupted."        )    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    # Install dependencies (smart installation - only installs missing packages)    # Load the setup module directly to avoid triggering other package imports    setup = load_setup_module(repo_path)        result = setup.smart_install_dependencies(        repo_path=repo_path,        include_dev=False,        verbose=True,    )    # Fail loudly if critical packages failed to install    if result["failed"]:        print(f"\n⚠️  WARNING: {len(result['failed'])} packages failed to install:")        for pkg in result["failed"]:            print(f"  - {pkg}")        print("\nYou may encounter import errors. Please check your internet connection.")    print("\n" + "=" * 70)    print("✅ Environment setup complete! You can now proceed with the notebook.")    print("=" * 70)else:    print("💻 Running in local environment. Configuring...\n")    # Find the repository root    repo_path = find_repo_root(Path.cwd())    if repo_path is None:        raise FileNotFoundError(            "Could not find repository root (no pyproject.toml found). "            "Please ensure you are running this notebook from within the "            "ADH-LLM-Tutorials-2025 repository directory."        )    print(f"✅ Found repository root: {repo_path}")    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    print("\n" + "=" * 70)    print("✅ Local environment configured successfully!")    print("=" * 70)    print("\n⚠️  Please ensure you have run: pip install -e .[dev]")    print("   (Required for local development)")

# 10 - Model Inference: Using a Trained Model for Predictions

## From Training to Production

In the previous notebooks, we trained and evaluated various sequence models for sepsis prediction. But training a model is only half the story—to be useful in a clinical setting, we need to be able to load a trained model and use it to make predictions on new, unseen patient data.

This notebook demonstrates the **inference pipeline**: how to save a trained model, load it in a separate environment (simulating production), and use it to generate risk predictions for new patients.

### What You'll Learn

1. How to properly save and load PyTorch model state dictionaries
2. How to load the saved feature scaler to ensure consistent preprocessing
3. How to create a complete inference function that encapsulates the entire prediction pipeline
4. The importance of strict contracts: preprocessing must match training exactly

### Key Concept: Training vs. Inference

- **Training**: We have labels, we compute loss, we update weights, we track metrics
- **Inference**: We have *no labels*, we only do a forward pass, we return predictions

The inference environment needs:
1. The trained model weights (state_dict)
2. The model architecture (our Python class)
3. The saved feature scaler (to preprocess new data the same way)
4. The preprocessing logic (imputation, scaling, tensor conversion)

In [ ]:
# Import required libraries
from pathlib import Path

import pandas as pd
import torch
import yaml

from core.config import GRUConfig
from core.data.physionet_sepsis import load_feature_preprocessor
from core.inference import prepare_patient_batch
from core.models import GRUModel
from core.notebook import ensure_project_root

## Step 1: Load the Production Artifacts

In a real deployment scenario, we would have three artifacts:

1. **Model weights** (`gru_best.pt`): The trained parameters
2. **Model configuration** (`gru.yaml`): The architecture specification
3. **Feature scaler** (`feature_scaler.json`): The fitted preprocessing parameters

These three files are all we need to recreate the exact model and preprocessing pipeline used during training.

In [ ]:
project_root = ensure_project_root()

# Define paths to the saved artifacts
model_path = Path("models/gru_best.pt")
config_path = Path("configs/gru.yaml")
scaler_path = Path("data/processed/feature_scaler.json")

# Verify all artifacts exist
if not model_path.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found at {model_path}. "
        "Please run notebook 02 (training) first."
    )
if not config_path.exists():
    raise FileNotFoundError(f"Config file not found at {config_path}.")
if not scaler_path.exists():
    raise FileNotFoundError(
        f"Feature scaler not found at {scaler_path}. "
        "Please run notebook 01 (data preparation) first."
    )

print("✅ All production artifacts found:")
print(f"   • Model weights: {model_path}")
print(f"   • Model config:  {config_path}")
print(f"   • Feature scaler: {scaler_path}")

In [ ]:
# Load the model configuration
config_dict = yaml.safe_load(config_path.read_text())
model_config = GRUConfig(**config_dict["model"])

print("Model Configuration:")
print(model_config)

# Instantiate the model architecture
inference_model = GRUModel(model_config)

# Load the trained weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_dict = torch.load(model_path, map_location=device)
inference_model.load_state_dict(state_dict)

# Set model to evaluation mode (disables dropout, etc.)
inference_model.eval()
inference_model.to(device)

print(f"\n✅ Model loaded and ready for inference on {device}")

In [ ]:
# Load the feature scaler and stored medians
scaler, feature_medians = load_feature_preprocessor(scaler_path)

print("✅ Feature scaler loaded successfully")
print(f"   Scaler was fit on {scaler.n_features_in_} features")
print(f"   Feature names: {list(scaler.feature_names_in_[:5])}...")
print("   Stored medians ensure inference preprocessing matches training.")

## Step 2: Create an Inference Function

This function encapsulates the **entire prediction pipeline** for a single patient:

1. **Input**: Raw patient time series as a DataFrame (same format as training data)
2. **Preprocessing**: Imputation → Scaling → Tensor conversion (must match training exactly)
3. **Inference**: Forward pass through the model
4. **Output**: A single sepsis risk probability

### Critical: Preprocessing Contract

The preprocessing steps **must be identical** to what was done during training:
- Same imputation strategy (forward fill + backward fill)
- Same feature scaling (using the saved scaler parameters)
- Same feature ordering

Any deviation will cause **distribution shift** and degrade model performance.

### Preserve the Time Column

Our preprocessing helper sorts each patient's timeline using the `ICULOS` column (hours since ICU admission).
Do **not** drop this column when selecting features for inference—`prepare_patient_batch()` will raise
an informative error if the column is missing so that predictions always follow the same chronological order
used during training.


In [ ]:
def predict_sepsis_risk(
    patient_df: pd.DataFrame,
    model: torch.nn.Module,
    scaler,
    feature_medians: dict[str, float],
    device: torch.device,
    time_column: str = "ICULOS",
) -> float:
    """
    Predict sepsis risk for a single patient's time series.

    This function encapsulates the entire inference pipeline by delegating
    preprocessing to the shared core helper (imputation + scaling) and running
    the model forward pass under a no-grad context.

    Parameters
    ----------
    patient_df : pd.DataFrame
        Raw patient time series data. Must contain all feature columns.
        Each row is one hour of ICU monitoring and must include the
        time-ordering column.
    model : torch.nn.Module
        The trained model in evaluation mode.
    scaler : StandardScaler
        The fitted feature scaler from training.
    feature_medians : dict[str, float]
        Stored feature medians computed during training.
    device : torch.device
        Device to run inference on (cpu or cuda).
    time_column : str, default="ICULOS"
        Column used to sort the patient's timeline before preprocessing.

    Returns
    -------
    float
        Predicted sepsis risk probability in range [0, 1].
    """
    if time_column not in patient_df.columns:
        raise KeyError(
            "Patient dataframe must include the time column used for sorting: "
            f"{time_column}."
        )

    features_tensor, mask = prepare_patient_batch(
        patient_df=patient_df,
        scaler=scaler,
        feature_medians=feature_medians,
        time_column=time_column,
    )

    features_tensor = features_tensor.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        logit = model(features_tensor, mask)
        probability = torch.sigmoid(logit).item()

    return probability


print("✅ Inference function defined")

## Step 3: Run a Test Prediction

Let's load a real patient's time series from the raw data and generate a sepsis risk prediction.

In a production system, this data would come from the hospital's Electronic Health Record (EHR) system in real-time.

In [ ]:
# Load a sample patient's data from the raw files
test_patient_path = Path("data/raw/physionet_2019/training/training_setA/p000001.psv")

if not test_patient_path.exists():
    raise FileNotFoundError(
        f"Test patient file not found at {test_patient_path}. "
        "Please ensure the PhysioNet data has been downloaded."
    )

# Read the patient data
patient_data = pd.read_csv(test_patient_path, sep="|", na_values="NaN")

print(f"✅ Loaded test patient data: {len(patient_data)} hourly measurements")
print(f"   Patient age: {patient_data['Age'].iloc[0]:.1f} years")
print(f"   ICU stay duration: {patient_data['ICULOS'].max()} hours")
true_status = "Positive" if patient_data["SepsisLabel"].max() == 1 else "Negative"
print(f"   True sepsis status: {true_status}")

In [ ]:
# Generate sepsis risk prediction
risk_score = predict_sepsis_risk(
    patient_df=patient_data,
    model=inference_model,
    scaler=scaler,
    feature_medians=feature_medians,
    device=device,
)

# Display the result
print("\n" + "=" * 60)
print("🏥 SEPSIS RISK ASSESSMENT")
print("=" * 60)
print("   Patient ID: p000001")
print(f"   Predicted Sepsis Risk: {risk_score*100:.2f}%")
print("=" * 60)

# Interpret the risk level
if risk_score < 0.3:
    risk_level = "LOW"
    color = "🟢"
elif risk_score < 0.7:
    risk_level = "MODERATE"
    color = "🟡"
else:
    risk_level = "HIGH"
    color = "🔴"

print(f"\n{color} Risk Level: {risk_level}")

if risk_level == "HIGH":
    print("   ⚠️  Consider clinical intervention and closer monitoring.")
elif risk_level == "MODERATE":
    print("   ⚡ Elevated risk - continue close monitoring.")
else:
    print("   ✓  Low risk - continue standard monitoring.")

## Step 4: Batch Inference for Multiple Patients

In practice, we might want to generate predictions for multiple patients at once. Let's demonstrate batch inference.

In [ ]:
# Load several test patients
test_patients_dir = Path("data/raw/physionet_2019/training/training_setA")
patient_files = sorted(test_patients_dir.glob("p00000[1-5].psv"))

print("Running batch inference on 5 test patients...\n")

results = []
for patient_file in patient_files:
    patient_id = patient_file.stem
    patient_data = pd.read_csv(patient_file, sep="|", na_values="NaN")

    # Generate prediction
    risk = predict_sepsis_risk(
        patient_data, inference_model, scaler, feature_medians, device
    )

    # Get ground truth
    true_label = int(patient_data["SepsisLabel"].max())

    results.append(
        {
            "Patient ID": patient_id,
            "Predicted Risk": f"{risk*100:.1f}%",
            "True Label": "Sepsis" if true_label == 1 else "No Sepsis",
            "Risk Score": risk,
        }
    )

# Display results as a table
results_df = pd.DataFrame(results)
display_df = results_df[["Patient ID", "Predicted Risk", "True Label"]]
print(display_df.to_string(index=False))

print("\n✅ Batch inference complete")

## Key Takeaways

### 1. The Inference Contract

For a model to work reliably in production, we must maintain a **strict contract** between training and inference:

- **Same preprocessing**: Imputation strategy, scaling parameters, feature ordering
- **Same model architecture**: Exact same hyperparameters (hidden_size, num_layers, etc.)
- **Same input format**: Data must be provided in the expected structure

Any deviation will cause **distribution shift** and unpredictable behavior.

### 2. Artifacts for Production

To deploy a model, you need exactly three things:

1. **Model weights** (`.pt` file): The trained parameters
2. **Model config** (`.yaml` file): Architecture specification
3. **Preprocessing artifacts** (`.json` scaler): Feature transformation parameters

With these three files and the model class definition, you can recreate the exact inference pipeline anywhere.

### 3. The Inference Function Pattern

The `predict_sepsis_risk()` function demonstrates the standard pattern for production inference:

```python
def predict(raw_data, model, preprocessor, device):
    # 1. Preprocess
    processed = preprocess(raw_data)
    
    # 2. Convert to tensors
    tensor = to_tensor(processed)
    
    # 3. Run model
    with torch.no_grad():
        output = model(tensor)
    
    # 4. Post-process
    return post_process(output)
```

This pattern ensures consistency and makes the inference logic easy to test.

### 4. Clinical Deployment Considerations

In a real clinical deployment, you would also need:

- **Input validation**: Ensure data meets expected format and ranges
- **Error handling**: Graceful degradation when data is missing or corrupted
- **Logging**: Track all predictions for audit trails
- **Monitoring**: Detect distribution drift and model degradation over time
- **Uncertainty quantification**: Provide confidence intervals, not just point estimates
- **Human-in-the-loop**: Model predictions should support, not replace, clinical judgment

## 🎯 Challenge Exercise

Try extending the inference pipeline:

1. **Add uncertainty quantification**: Use Monte Carlo dropout to generate confidence intervals
2. **Create a REST API**: Wrap the inference function in a Flask or FastAPI endpoint
3. **Add input validation**: Use Pydantic to validate incoming patient data
4. **Implement model monitoring**: Log predictions and compute drift metrics
5. **Compare models**: Load both GRU and LSTM models and compare their predictions

Can you build a production-ready inference service that would be safe for clinical use?